# SBOM-to-Audit — Stage 3 Colab Checkpoint

This notebook independently verifies package v0.3.0 and the Stage 3 scope-aware pilot
from a clean GitHub checkout and an isolated Python environment.

It verifies all committed scenarios, strict source validation, the complete release gate,
deterministic outputs, Ghost-Logger conflict lifecycle, False Comfort scope mismatch,
the matched negative control, and machine-generated pilot assets.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"  # Replace with the exact Stage 3 branch or tag when available.
print("Repository:", REPO_URL)
print("Reference:", REF)


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/sbom_to_audit")
VENV = Path("/content/sbom_to_audit_stage3_venv")
for path in (WORKDIR, VENV):
    if path.exists():
        shutil.rmtree(path)
clone_command = [
    "git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)
]
subprocess.run(clone_command, check=True)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
print("Commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())


In [ ]:
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
except subprocess.CalledProcessError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "virtualenv"], check=True
    )
    subprocess.run(
        [sys.executable, "-m", "virtualenv", str(VENV)], check=True
    )
PYTHON = VENV / "bin" / "python"
upgrade_command = [
    str(PYTHON), "-m", "pip", "install", "--upgrade",
    "pip", "setuptools", "wheel"
]
subprocess.run(upgrade_command, check=True)
install_command = [
    str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"
]
subprocess.run(install_command, check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")


In [ ]:
import json

REPORT = Path("/content/stage3_colab_release_validation.json")
release_command = [
    str(PYTHON), "scripts/release_check.py", "--report", str(REPORT)
]
subprocess.run(release_command, check=True)
release = json.loads(REPORT.read_text(encoding="utf-8"))
assert release["status"] == "PASS", release["errors"]
assert len(release["deterministic_hashes"]) == 18
print("PASS: canonical multi-scenario release gate")
print("Deterministic outputs:", len(release["deterministic_hashes"]))


In [ ]:
OUTPUTS = Path("/content/stage3_checkpoint_outputs")
ASSETS = Path("/content/stage3_checkpoint_assets")
for path in (OUTPUTS, ASSETS):
    if path.exists():
        shutil.rmtree(path)
for scenario in sorted(Path("data/scenarios").glob("*.yaml")):
    command = [
        str(PYTHON), "-m", "sbom_to_audit.cli",
        "--scenario", str(scenario), "--output-root", str(OUTPUTS)
    ]
    subprocess.run(command, check=True)
asset_command = [
    str(PYTHON), "paper_assets/scripts/build_stage3_assets.py",
    "--output-root", str(OUTPUTS), "--destination", str(ASSETS)
]
subprocess.run(asset_command, check=True)
print("PASS: checkpoint outputs and pilot assets generated")


In [ ]:
import csv

def pack(name):
    path = OUTPUTS / "evidence_packs" / f"{name}.json"
    return json.loads(path.read_text(encoding="utf-8"))

def conflicts(name):
    path = OUTPUTS / "conflict_reports" / f"{name}.json"
    return json.loads(path.read_text(encoding="utf-8"))

def states(name):
    path = OUTPUTS / "state_logs" / f"{name}.csv"
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))

ghost = pack("ghost_logger")
ghost_conflicts = conflicts("ghost_logger")
assert ghost["decision_state"]["recommended_state"] == "Report-Ready"
assert ghost["orchestration_metrics"]["C_t"] is False
assert ghost_conflicts["resolved_conflicts"] == 1
assert ghost_conflicts["active_conflicts"] == 0

fc = pack("false_comfort")
fc_states = states("false_comfort")
expected_fc_states = [
    "Monitor", "Report-Ready", "Report-Ready", "Report-Ready"
]
assert [row["observed_state"] for row in fc_states] == expected_fc_states
assert fc["supplier_assertions"]["asserted_csaf_vex_status"] == (
    "known_not_affected"
)
assert fc["supplier_assertions"]["scope_applicability"] == "scope_mismatch"
assert conflicts("false_comfort")["detected_conflicts"] == 0

control = pack("false_comfort_control")
assert control["supplier_assertions"]["scope_applicability"] == "applicable"
assert control["decision_state"]["recommended_state"] == "Document No-Report"
assert control["orchestration_metrics"]["A_t"] == 0.1

print("PASS: Ghost-Logger conflict lifecycle")
print("PASS: False Comfort scope mismatch and trajectory")
print("PASS: scope-matched negative control")


In [ ]:
from IPython.display import SVG, display

figure = ASSETS / "figures" / "false_comfort_scope_comparison.svg"
display(SVG(filename=str(figure)))


In [ ]:
import datetime as dt
import hashlib
import zipfile

ENV = Path("/content/stage3_colab_environment.json")
environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "commit": COMMIT,
    "kernel_python": sys.version,
    "isolated_python": subprocess.check_output(
        [str(PYTHON), "--version"], text=True
    ).strip(),
    "platform": platform.platform(),
    "release_status": release["status"],
}
ENV.write_text(
    json.dumps(environment, indent=2) + "\n", encoding="utf-8"
)
BUNDLE = Path("/content/stage3_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT, REPORT.name)
    archive.write(ENV, ENV.name)
    for root in (OUTPUTS, ASSETS):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, f"{root.name}/{path.relative_to(root)}")
digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("SHA-256:", digest)
print("Preserve this hash with the exact Git commit above.")
